# SSL Experiment — nb01: SimCLR Baseline

**Goal:** Train a SimCLR model on CIFAR-10 (no labels) and establish baseline metrics for comparison against CTLS-SSL.

**Method:** NT-Xent loss on circuit embeddings z1, z2 from two augmented views of the same image. No circuit consistency loss — this is the pure SSL baseline.

**Outputs:**
- Linear probe accuracy (frozen backbone, 100-epoch linear head)
- KNN-20 accuracy in circuit embedding space
- UMAP of circuit embeddings colored by true CIFAR-10 class
  (shows what cluster structure emerged without any label supervision)

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml

with open('configs/ssl/simclr.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

## 1. Train SimCLR

Expected training time: ~35–50 min on GPU (200 epochs, batch_size=256).

**Skip if `experiments/ssl/simclr/best.pt` already exists.**

In [ ]:
from training.ssl_trainer import SSLTrainer

trainer = SSLTrainer(config)
trainer.train()

## 2. Load Model

In [ ]:
import torch
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_ssl_model(config, ckpt_path, device):
    mcfg = config['model']
    ecfg = mcfg['meta_encoder']
    smcfg = mcfg.get('soft_mask', {})

    soft_mask = SoftMask(init_temperature=smcfg.get('init_temperature', 1.0))
    backbone = CTLSBackbone(
        arch=mcfg['arch'],
        num_classes=mcfg['num_classes'],
        soft_mask=soft_mask,
        pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()
    return backbone, meta_encoder, ckpt

backbone, meta_encoder, ckpt = load_ssl_model(
    config, 'experiments/ssl/simclr/best.pt', DEVICE
)
print(f"Loaded SimCLR checkpoint (epoch {ckpt['epoch']})")

## 3. Linear Probe Accuracy

Standard SSL evaluation: freeze the backbone, train a single linear layer on CIFAR-10 training labels for 100 epochs, report test accuracy.

In [ ]:
from data.cifar import get_standard_loaders
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

dcfg = config['data']
train_loader_labeled, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=256,
    num_workers=4,
    augment=True,
    download=True,
)

# --- Collect all train embeddings (frozen backbone) ---
backbone.eval(); meta_encoder.eval()
all_z_train, all_y_train = [], []
with torch.no_grad():
    for x, y in train_loader_labeled:
        _, traj = backbone(x.to(DEVICE))
        z = meta_encoder(traj)
        all_z_train.append(z.cpu())
        all_y_train.append(y)
all_z_train = torch.cat(all_z_train)
all_y_train = torch.cat(all_y_train)

all_z_val, all_y_val = [], []
with torch.no_grad():
    for x, y in val_loader:
        _, traj = backbone(x.to(DEVICE))
        z = meta_encoder(traj)
        all_z_val.append(z.cpu())
        all_y_val.append(y)
all_z_val = torch.cat(all_z_val)
all_y_val = torch.cat(all_y_val)

print(f"Train embeddings: {all_z_train.shape} | Val embeddings: {all_z_val.shape}")

In [ ]:
# Train linear probe on frozen embeddings
from torch.utils.data import TensorDataset, DataLoader as DL

EMBEDDING_DIM = config['model']['meta_encoder']['embedding_dim']
linear_probe = nn.Linear(EMBEDDING_DIM, 10).to(DEVICE)
probe_opt = AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=1e-4)
probe_ds = DL(TensorDataset(all_z_train, all_y_train), batch_size=256, shuffle=True)

for ep in range(100):
    linear_probe.train()
    for z, y in probe_ds:
        z, y = z.to(DEVICE), y.to(DEVICE)
        loss = F.cross_entropy(linear_probe(z), y)
        probe_opt.zero_grad(); loss.backward(); probe_opt.step()

linear_probe.eval()
with torch.no_grad():
    preds = linear_probe(all_z_val.to(DEVICE)).argmax(dim=-1).cpu()
linear_probe_acc = (preds == all_y_val).float().mean().item()
print(f"Linear probe accuracy (SimCLR): {linear_probe_acc:.4f}")

## 4. KNN-20 Accuracy

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

knn = KNeighborsClassifier(n_neighbors=20, metric='cosine')
knn.fit(all_z_train.numpy(), all_y_train.numpy())
knn_preds = knn.predict(all_z_val.numpy())
knn_acc = (knn_preds == all_y_val.numpy()).mean()
print(f"KNN-20 accuracy (SimCLR): {knn_acc:.4f}")

## 5. UMAP: Circuit Embeddings Colored by True Class

No class labels were used during training. The UMAP shows what semantic structure
the model discovered purely from augmentation-invariance pressure.

In [ ]:
from evaluation.circuit_viz import CircuitVisualizer
import matplotlib.pyplot as plt

viz = CircuitVisualizer(backbone, meta_encoder, val_loader, DEVICE)
fig = viz.plot_umap(
    title='SimCLR — circuit embedding space (no labels used in training)',
    compare_output=False,
    max_samples=5000,
)
os.makedirs('experiments/ssl/simclr', exist_ok=True)
fig.savefig('experiments/ssl/simclr/umap_circuit.png', dpi=150, bbox_inches='tight')
plt.show()

scores = viz.cluster_separation_score(max_samples=5000)
print(f"\nCircuit silhouette (SimCLR): {scores['silhouette_circuit']:.4f}")
print(f"Output  silhouette (SimCLR): {scores['silhouette_output']:.4f}")

## Summary

| Metric | SimCLR |
|--------|--------|
| Linear probe accuracy | ___ |
| KNN-20 accuracy | ___ |
| Circuit space silhouette | ___ |

**Next:** Run `nb02_dino_baseline.ipynb` then `nb03_ctls_ssl_training.ipynb`.